# 6b. Expert-Phase Optogenetic Inactivation Analysis

**Question**: Is PPC dispensable during expert performance?

**Prediction**: PPC inactivation has NO effect on expert-phase behaviour.
This is a null prediction — it must be supported by equivalence testing
(TOST), not merely by failing to reject the standard null.

**Design**: VGAT-ChR2-EYFP × Zapit bilateral PPC silencing, 30%
interleaved opto trials (choice window). Between-subjects genotype:
Het (expressing) vs WT (light-only control). Within-subjects phases:
baseline → masking → opto → washout.

---

### Pooling levels used in this notebook

| Level | What's pooled | Where used | Why |
|:------|:-------------|:-----------|:----|
| **Trial-level, within-session** | Opto vs control within one session (~90 vs ~210 trials) | Per-session effect sizes | Primary comparison unit |
| **Trial-level, within-animal-phase** | All sessions of one phase for one animal (e.g. 5 opto sessions → ~450 opto + ~1050 control) | Psychometric fits, UMs | Single sessions underpowered for curve fitting |
| **Session-level, per-animal** | One stat per session, track across sessions | Phase stability | Tests for drift |
| **Animal-level, cohort** | One mean effect per animal → t-test / TOST | Equivalence testing | Avoids pseudoreplication |

---

### Structure

1. **Hook** — one example animal, full picture
2. **Controls** — masking, WT, baseline drift (must pass before interpreting)
3. **Cohort landscape** — all animals, forest plot
4. **The punchline** — TOST equivalence testing
5. **Between-animal** — genotype, model assignment
6. **Robustness** — stimulus-conditional accuracy, RT, no-response, bias
7. **Appendix** — full per-animal reports

In [ ]:
%matplotlib inline
from shared_setup import *

from analysis.opto import (
    OptoPhase, assign_opto_phases,
    opto_relative_mask, split_trials_by_opto,
    get_post_opto_mask, extract_trial_arrays,
    within_session_effect, phase_pooled_comparison, compute_opto_um,
    phase_stability, genotype_interaction,
    animal_opto_report, cohort_opto_report,
    opto_by_model_assignment, expert_null_test, expert_um_test,
    phase_opto_interaction, _session_has_opto,
)
from plotting.opto import (
    plot_opto_psychometric, plot_phase_trajectory,
    plot_opto_um_comparison, plot_phase_stability,
    plot_within_session_summary, plot_genotype_interaction,
    plot_model_assignment_effects, plot_equivalence_test,
    plot_phase_interaction, plot_animal_opto_report,
    OPTO_COLOUR, CTRL_COLOUR, POST_OPTO_COLOUR, PHASE_COLOURS,
)
from behav_utils.plotting.styles import apply_style, UM_CMAP
from behav_utils.analysis.comparison import compare_conditions
apply_style()

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

params = ['accuracy', 'pse', 'slope', 'lapse_low', 'lapse_high']

In [ ]:
experiment, info = load_data()
print(f"Mode: {info['mode']}")
print(f"Animals: {experiment.n_animals}")
print(f"Total sessions: {sum(a.n_sessions for a in experiment.animals.values())}")

In [ ]:
# ── Identify opto animals and assign phases ──────────────────────────
all_phases = {}
opto_animals = {}

for aid, animal in experiment.animals.items():
    phases = assign_opto_phases(animal)
    all_phases[aid] = phases
    if OptoPhase.EXPERT_OPTO in phases:
        opto_animals[aid] = animal

het_animals = {a: v for a, v in opto_animals.items()
               if getattr(v, 'genotype', 'unknown') == 'het'}
wt_animals  = {a: v for a, v in opto_animals.items()
               if getattr(v, 'genotype', 'unknown') == 'wt'}

print(f"Opto animals: {len(opto_animals)} ({len(het_animals)} het, {len(wt_animals)} wt)")
for aid in sorted(opto_animals):
    ph = all_phases[aid]
    geno = getattr(opto_animals[aid], 'genotype', '?')
    counts = {p.value.split('_', 1)[-1]: sum(1 for x in ph if x == p)
              for p in [OptoPhase.EXPERT_BASELINE, OptoPhase.MASKING,
                        OptoPhase.EXPERT_OPTO, OptoPhase.EXPERT_WASHOUT]}
    parts = ', '.join(f"{k}={v}" for k, v in counts.items() if v > 0)
    print(f"  {aid} ({geno}): {parts}")

In [ ]:
# ── Sanity: opto trial fractions (expect ~30%) ───────────────────────
for aid in sorted(opto_animals):
    ph = all_phases[aid]
    for s, p in zip(opto_animals[aid].sessions, ph):
        if p != OptoPhase.EXPERT_OPTO:
            continue
        om, cm = split_trials_by_opto(s)
        total = om.sum() + cm.sum()
        frac = om.sum() / total if total > 0 else 0
        flag = '' if 0.2 < frac < 0.4 else '  ⚠️'
        print(f"  {aid} {s.session_id}: {om.sum()}/{total} = {frac:.0%}{flag}")

In [ ]:
# ── Pick example animal (het with most opto sessions) ────────────────
pool = het_animals if het_animals else opto_animals
EXAMPLE_ID = max(pool, key=lambda a: sum(
    1 for p in all_phases[a] if p == OptoPhase.EXPERT_OPTO))
example = experiment.animals[EXAMPLE_ID]
ex_phases = all_phases[EXAMPLE_ID]
print(f"Example: {EXAMPLE_ID} ({getattr(example, 'genotype', '?')})")

---
## 1. Hook — example animal

Everything about one animal before any cohort statistics.

Three conditions on one psychometric curve answer two questions at once:
- **Masking (grey, dashed)** ≈ **Control (blue)**? → light is inert
- **Opto ON (red)** ≈ **Control (blue)**? → PPC is dispensable

If all three overlap, PPC inactivation has no measurable effect.

In [ ]:
# ── Phase trajectory (Full_Task_Cont only, WITH legend) ──────────────
ftc = [(s, p) for s, p in zip(example.sessions, ex_phases)
       if s.metadata.get('stage') == 'Full_Task_Cont']
if not ftc:
    ftc = list(zip(example.sessions, ex_phases))

ftc_s, ftc_p = zip(*ftc)
fig = plot_phase_trajectory(list(ftc_s), list(ftc_p), stat_name='accuracy',
    title=f'{EXAMPLE_ID} — accuracy across phases')

# Add legend (plot_phase_trajectory doesn't include one)
from matplotlib.lines import Line2D
legend_phases = [
    (OptoPhase.EXPERT_BASELINE, 'Baseline'),
    (OptoPhase.MASKING, 'Masking'),
    (OptoPhase.EXPERT_OPTO, 'Opto'),
    (OptoPhase.EXPERT_WASHOUT, 'Washout'),
]
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=PHASE_COLOURS[p],
           markersize=8, label=lab) for p, lab in legend_phases if p in set(ftc_p)]
fig.axes[0].legend(handles=handles, loc='lower right', fontsize=8)
plt.show()

In [ ]:
# ── Three-condition psychometric overlay ─────────────────────────────
# One figure answers two questions at once:
#   Masking ≈ Control?  → light is inert
#   Opto ON ≈ Control?  → PPC is dispensable
#
# Masking light-on = opto_on trials from masking sessions (light, no activation)
# Opto ON          = opto_on trials from real opto sessions (PPC silenced)
# Control          = opto_off trials from opto sessions (no light)

opto_sess = [s for s, p in zip(example.sessions, ex_phases)
             if p == OptoPhase.EXPERT_OPTO]
mask_sess = [s for s, p in zip(example.sessions, ex_phases)
             if p == OptoPhase.MASKING]

print(f"Opto sessions: {len(opto_sess)}, Masking sessions: {len(mask_sess)}")

conditions = [
    ('Control (opto OFF)',   opto_sess, 'control',  CTRL_COLOUR,       '-'),
    ('Opto ON (PPC silenced)', opto_sess, 0,        OPTO_COLOUR,       '-'),
    ('Masking (light only)', mask_sess,  0,          '#999999',         '--'),
]

fig, ax = plt.subplots(1, 1, figsize=(6, 4.5))
x_fine = np.linspace(-1, 1, 200)

for label, sess_list, delta, colour, ls in conditions:
    if not sess_list:
        continue
    all_stim, all_ch = [], []
    for s in sess_list:
        mask = opto_relative_mask(s, delta=delta)
        arr = extract_trial_arrays(s, mask)
        if arr is None:
            continue
        v = ~arr['no_response']
        all_stim.append(arr['stimuli'][v])
        all_ch.append(arr['choices'][v])

    if not all_stim:
        continue

    stim = np.concatenate(all_stim)
    choice = np.concatenate(all_ch)

    # Binned data
    bins = np.linspace(-1, 1, 9)
    centres = (bins[:-1] + bins[1:]) / 2
    means = [np.nanmean(choice[(stim >= bins[b]) & (stim < bins[b+1])])
             if ((stim >= bins[b]) & (stim < bins[b+1])).sum() > 0 else np.nan
             for b in range(8)]
    ax.plot(centres, means, 'o', color=colour, markersize=5, alpha=0.6)

    # Fitted curve
    try:
        pfit = fit_psychometric(stim, choice)
        y = cumulative_gaussian(x_fine, pfit['mu'], pfit['sigma'],
                                pfit['lapse_low'], pfit['lapse_high'])
        ax.plot(x_fine, y, color=colour, ls=ls, lw=1.5,
                label=f"{label} (n={len(stim)})")
    except Exception:
        pass

ax.axhline(0.5, ls=':', color='grey', alpha=0.3)
ax.axvline(0.0, ls=':', color='grey', alpha=0.3)
ax.set_xlabel('Stimulus'); ax.set_ylabel('P(choose B)')
ax.set_xlim(-1.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8, loc='lower right')
ax.set_title(f'{EXAMPLE_ID} — masking vs opto vs control')
fig.tight_layout(); plt.show()

In [ ]:
# ── Three-condition update matrices ──────────────────────────────────
# Same three conditions as the psychometric overlay.

um_conditions = [
    ('Control',  opto_sess, 'control'),
    ('Opto ON',  opto_sess, 0),
    ('Masking',  mask_sess, 0),
]

ums_3 = {}
for label, sess_list, delta in um_conditions:
    if sess_list:
        um, _, info = compute_opto_um(sess_list, delta=delta)
    else:
        um = np.full((8, 8), np.nan)
        info = {'n_trials': 0}
    ums_3[label] = um
    nt = info.get('n_trials', 0) if isinstance(info, dict) else 0
    print(f"  {label:>10s}: {nt} trials")

n_panels = 3 + (1 if len(opto_sess) > 0 else 0)
fig, axes = plt.subplots(1, n_panels, figsize=(4.5 * n_panels, 4))

vmax = max(np.nanmax(np.abs(v)) for v in ums_3.values()
           if not np.all(np.isnan(v)))
vmax = max(vmax, 0.01)

for ax, (label, um) in zip(axes[:3], ums_3.items()):
    im = ax.imshow(um, cmap=UM_CMAP, vmin=-vmax, vmax=vmax,
                   origin='lower', aspect='equal')
    ax.set_title(label); ax.set_xlabel('Previous stimulus')
    plt.colorbar(im, ax=ax, fraction=0.046)
axes[0].set_ylabel('Current stimulus')

if n_panels > 3:
    diff = ums_3['Opto ON'] - ums_3['Control']
    vmax_d = max(np.nanmax(np.abs(diff)), 0.01)
    im = axes[3].imshow(diff, cmap=UM_CMAP, vmin=-vmax_d, vmax=vmax_d,
                        origin='lower', aspect='equal')
    axes[3].set_title('Opto ON − Control')
    axes[3].set_xlabel('Previous stimulus')
    plt.colorbar(im, ax=axes[3], fraction=0.046)

fig.suptitle(f'{EXAMPLE_ID} — update matrices (3 conditions)', fontsize=12, y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
# ── Per-session opto effects (all 5 params) ──────────────────────────
ex_within = []
for idx, (s, p) in enumerate(zip(example.sessions, ex_phases)):
    if p != OptoPhase.EXPERT_OPTO:
        continue
    eff = within_session_effect(s)
    if eff is not None:
        ex_within.append({'phase': p, 'session_idx': idx, 'effect': eff})

fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, param in zip(axes, params):
    vals = [e['effect']['diff'][param] for e in ex_within]
    colours = [OPTO_COLOUR if abs(v) > 0.02 else '#AAAAAA' for v in vals]
    ax.bar(range(len(vals)), vals, color=colours, alpha=0.7, edgecolor='white')
    ax.axhline(0, ls='-', color='grey', alpha=0.3)
    ax.set_title(param); ax.set_xlabel('Opto session')
    if ax == axes[0]:
        ax.set_ylabel('Δ (opto − control)')
fig.suptitle(f'{EXAMPLE_ID} — per-session opto effects', y=1.02)
fig.tight_layout(); plt.show()

---
## 2. Controls

Four checks that must pass before opto effects are interpretable.
Each asks a different question:

| Control | Question | Comparison | Expected |
|:--------|:---------|:-----------|:---------|
| **2a. Masking within-session** | Does light alone affect behaviour? | Light-on vs light-off within masking sessions | Null |
| **2b. Masking vs expert baseline** | Does the light procedure change control-trial behaviour? | Masking-session control trials vs expert baseline sessions (no light) | Null |
| **2c. WT within-session** | Does light in a non-expressing animal do anything? | Opto-on vs opto-off within WT expert sessions | Null |
| **2d. Het baseline vs opto control** | Do control trials drift when opto sessions start? | Het expert baseline sessions vs het opto-phase control trials | Null |

If any of these show a significant effect, downstream interpretation is compromised.

In [ ]:
# ── 2a. Masking within-session ───────────────────────────────────────
# Within masking sessions: light-on trials vs light-off trials.
# Both are non-activating (masking light only). Should be identical.

masking_effects = []
for aid in sorted(opto_animals):
    for s, p in zip(opto_animals[aid].sessions, all_phases[aid]):
        if p != OptoPhase.MASKING:
            continue
        eff = within_session_effect(s)
        if eff is not None:
            eff['animal_id'] = aid
            masking_effects.append(eff)

if masking_effects:
    print(f"Masking sessions: {len(masking_effects)}\n")
    print(f"{'Animal':>10s}" + ''.join(f"{p:>12s}" for p in params))
    print('-' * (10 + 12 * 5))
    for e in masking_effects:
        vals = ''.join(f"{e['diff'].get(p, np.nan):+12.4f}" for p in params)
        print(f"{e['animal_id']:>10s}{vals}")
else:
    print("⚠️  No masking sessions found.")
    print("   Check masking_sessions in config.yaml and session.masking attribute.")

In [ ]:
# ── Masking psychometric overlay (pooled across animals) ─────────────
mask_sess = [s for aid in opto_animals
             for s, p in zip(opto_animals[aid].sessions, all_phases[aid])
             if p == OptoPhase.MASKING]

if mask_sess:
    fig = plot_opto_psychometric(mask_sess, n_bootstrap=200, show_post_opto=False,
        title=f'Masking control — light-on vs light-off ({len(mask_sess)} sessions)')
    plt.show()
else:
    print("No masking sessions to plot.")

In [ ]:
# ── 2b. Masking vs expert baseline (per-animal overlaid psychometrics) ──
# Question: does the light PROCEDURE (not the light itself) change
# control-trial behaviour? Compare masking-session control trials
# vs expert baseline sessions (no opto hardware present).
#
# Expert baseline = expert_uniform preset, filtered to non-opto sessions.
# This avoids including early learning sessions.

from behav_utils.data.selection import select_sessions
from behav_utils.analysis.psychometry import fit_psychometric
from behav_utils.analysis.utils import cumulative_gaussian

x_fine = np.linspace(-1, 1, 200)

animals_with_masking = [aid for aid in opto_animals
                        if OptoPhase.MASKING in all_phases[aid]]

if animals_with_masking:
    n_animals = len(animals_with_masking)
    fig, axes = plt.subplots(1, min(n_animals, 6), figsize=(5 * min(n_animals, 6), 4),
                             squeeze=False)
    axes = axes.flatten()

    for i, aid in enumerate(sorted(animals_with_masking)[:6]):
        ax = axes[i]
        animal = opto_animals[aid]
        ph = all_phases[aid]

        # Expert baseline: expert_uniform preset, non-opto sessions only
        baseline_sess = select_sessions(animal, preset='expert_uniform')
        baseline_sess = [s for s in baseline_sess if not _session_has_opto(s)]

        masking_sess = [s for s, p in zip(animal.sessions, ph)
                        if p == OptoPhase.MASKING]

        for label, sess_list, colour, ls in [
            ('Expert baseline', baseline_sess, CTRL_COLOUR, '-'),
            ('Masking (ctrl trials)', masking_sess, '#AAAAAA', '--'),
        ]:
            all_stim, all_ch = [], []
            for s in sess_list:
                # Use control trials only (exclude opto-flagged trials)
                mask = opto_relative_mask(s, delta='control')
                arr = extract_trial_arrays(s, mask)
                if arr is None:
                    # For baseline sessions with no opto field, use all valid
                    valid = ~s.trials.abort
                    if valid.sum() < 10:
                        continue
                    ch = s.trials.choice[valid].astype(float)
                    v2 = ~np.isnan(ch)
                    all_stim.append(s.trials.stimulus[valid][v2])
                    all_ch.append(ch[v2])
                else:
                    v = ~arr['no_response']
                    all_stim.append(arr['stimuli'][v])
                    all_ch.append(arr['choices'][v])

            if not all_stim:
                continue

            stim = np.concatenate(all_stim)
            choice = np.concatenate(all_ch)

            try:
                pfit = fit_psychometric(stim, choice)
                y = cumulative_gaussian(x_fine, pfit['mu'], pfit['sigma'],
                                        pfit['lapse_low'], pfit['lapse_high'])
                ax.plot(x_fine, y, color=colour, ls=ls, label=f"{label} (n={len(stim)})")
            except Exception:
                pass

        ax.axhline(0.5, ls=':', color='grey', alpha=0.3)
        ax.axvline(0.0, ls=':', color='grey', alpha=0.3)
        ax.set_title(aid); ax.set_xlim(-1.05, 1.05); ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel('Stimulus'); ax.legend(fontsize=7)
        if i == 0:
            ax.set_ylabel('P(choose B)')

    fig.suptitle('2b: Masking vs expert baseline (control trials)', y=1.03)
    fig.tight_layout(); plt.show()
else:
    print("No animals with masking sessions.")

In [ ]:
# ── 2c. WT within-session ────────────────────────────────────────────
# Light is delivered to non-expressing animals. Should be null.

wt_effects = []
for aid in sorted(wt_animals):
    for s, p in zip(wt_animals[aid].sessions, all_phases[aid]):
        if p != OptoPhase.EXPERT_OPTO:
            continue
        eff = within_session_effect(s)
        if eff is not None:
            eff['animal_id'] = aid
            wt_effects.append(eff)

if wt_effects:
    print(f"WT opto sessions: {len(wt_effects)}\n")
    print(f"{'Animal':>10s}" + ''.join(f"{p:>12s}" for p in params))
    print('-' * (10 + 12 * 5))
    for e in wt_effects:
        vals = ''.join(f"{e['diff'].get(p, np.nan):+12.4f}" for p in params)
        print(f"{e['animal_id']:>10s}{vals}")

    wt_sess = [s for aid in wt_animals
               for s, p in zip(wt_animals[aid].sessions, all_phases[aid])
               if p == OptoPhase.EXPERT_OPTO]
    fig = plot_opto_psychometric(wt_sess, n_bootstrap=200, show_post_opto=True,
        title=f'WT — opto vs control ({len(wt_sess)} sessions)')
    plt.show()
else:
    print("No WT animals with expert opto data yet.")

In [ ]:
# ── 2d. Het baseline vs opto control trials ──────────────────────────
# Do control (non-opto) trials change when the opto phase starts?
# Compare expert baseline psychometric vs opto-phase control-trial psychometric.


het_with_baseline = [aid for aid in het_animals
                     if OptoPhase.EXPERT_BASELINE in all_phases[aid]
                     and OptoPhase.EXPERT_OPTO in all_phases[aid]]

if het_with_baseline:
    n_show = min(len(het_with_baseline), 6)
    fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4), squeeze=False)
    axes = axes.flatten()

    for i, aid in enumerate(sorted(het_with_baseline)[:n_show]):
        ax = axes[i]
        animal = het_animals[aid]
        ph = all_phases[aid]

        # Expert baseline (non-opto sessions, expert_uniform preset)
        bl_sess = select_sessions(animal, preset='expert_uniform')
        bl_sess = [s for s in bl_sess if not _session_has_opto(s)]

        # Opto-phase control trials
        opto_sess = [s for s, p in zip(animal.sessions, ph)
                     if p == OptoPhase.EXPERT_OPTO]

        for label, sess_list, colour, ls, use_ctrl_only in [
            ('Expert baseline', bl_sess, CTRL_COLOUR, '-', False),
            ('Opto phase (ctrl trials)', opto_sess, OPTO_COLOUR, '--', True),
        ]:
            all_stim, all_ch = [], []
            for s in sess_list:
                if use_ctrl_only:
                    mask = opto_relative_mask(s, delta='control')
                    arr = extract_trial_arrays(s, mask)
                    if arr is None:
                        continue
                    v = ~arr['no_response']
                    all_stim.append(arr['stimuli'][v])
                    all_ch.append(arr['choices'][v])
                else:
                    valid = ~s.trials.abort
                    ch = s.trials.choice[valid].astype(float)
                    v2 = ~np.isnan(ch)
                    if v2.sum() < 10:
                        continue
                    all_stim.append(s.trials.stimulus[valid][v2])
                    all_ch.append(ch[v2])

            if not all_stim:
                continue
            stim = np.concatenate(all_stim)
            choice = np.concatenate(all_ch)

            try:
                pfit = fit_psychometric(stim, choice)
                y = cumulative_gaussian(x_fine, pfit['mu'], pfit['sigma'],
                                        pfit['lapse_low'], pfit['lapse_high'])
                ax.plot(x_fine, y, color=colour, ls=ls,
                        label=f"{label} (n={len(stim)})")
            except Exception:
                pass

        ax.axhline(0.5, ls=':', color='grey', alpha=0.3)
        ax.axvline(0.0, ls=':', color='grey', alpha=0.3)
        ax.set_title(aid); ax.set_xlim(-1.05, 1.05); ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel('Stimulus'); ax.legend(fontsize=7)
        if i == 0:
            ax.set_ylabel('P(choose B)')

    fig.suptitle('2d: Het baseline vs opto-phase control trials', y=1.03)
    fig.tight_layout(); plt.show()
else:
    print("No het animals with both baseline and opto phases.")

In [ ]:
# ── Control verdicts ─────────────────────────────────────────────────
print("Control summary:")
print("=" * 60)

# 2a: masking
if masking_effects:
    max_acc_diff = max(abs(e['diff']['accuracy']) for e in masking_effects)
    print(f"  2a Masking within-session:  max |Δ accuracy| = {max_acc_diff:.3f}"
          f"  {'✓ PASS' if max_acc_diff < 0.05 else '⚠️ CHECK'}")
else:
    print("  2a Masking within-session:  no data")

# 2c: WT
if wt_effects:
    max_wt_diff = max(abs(e['diff']['accuracy']) for e in wt_effects)
    print(f"  2c WT within-session:      max |Δ accuracy| = {max_wt_diff:.3f}"
          f"  {'✓ PASS' if max_wt_diff < 0.05 else '⚠️ CHECK'}")
else:
    print("  2c WT within-session:      no data")

print()
print("If any control fails, downstream opto results are suspect.")

---
## 3. Cohort landscape

One report per animal → summary table → forest plot.

Animal-level pooling: each animal's expert opto sessions are pooled
(trial-level, within-animal-phase) for psychometric fits. Effect sizes
are per-session, then averaged per animal for the forest plot.

In [ ]:
# ── Run all animal reports (single computation, used throughout) ─────
reports = {}
for aid, animal in opto_animals.items():
    reports[aid] = animal_opto_report(animal)
print(f"Reports: {len(reports)} animals")

In [ ]:
# ── Summary table with significance indicators ──────────────────────
het_reports = {a: reports[a] for a in het_animals if a in reports}
wt_reports  = {a: reports[a] for a in wt_animals if a in reports}

print(f"{'Animal':>10s} {'Geno':>5s} {'nOpto':>6s} {'nCtrl':>6s}"
      + ''.join(f"{p:>12s}" for p in params)
      + f"{'UM_RMSE':>10s}")
print('-' * (10 + 5 + 6 + 6 + 12 * 5 + 10))

for aid in sorted(reports):
    r = reports[aid]
    comp = r['phase_comparisons'].get(OptoPhase.EXPERT_OPTO)
    if comp is None:
        continue
    geno = r['genotype']
    vals = ''.join(f"{comp['diff'].get(p, np.nan):+12.4f}" for p in params)
    rmse = comp.get('um_rmse', np.nan)
    print(f"{aid:>10s} {geno:>5s} {comp['n_opto']:>6d} "
          f"{comp['n_control']:>6d}{vals}{rmse:>10.4f}")

In [ ]:
# ── Forest plot: per-animal accuracy effect ± CI ─────────────────────
# Two CI options shown side by side:
#   Left: bootstrap CI from compare_conditions (honest but slow)
#   Right: SEM from per-session effects (fast approximation)

from analysis.opto import _collect_phase_effects

# Collect per-animal data
forest_data = []
for aid in sorted(het_reports):
    r = het_reports[aid]
    comp = r['phase_comparisons'].get(OptoPhase.EXPERT_OPTO)
    if comp is None:
        continue

    # Mean per-session effect for this animal
    effs = _collect_phase_effects(r, OptoPhase.EXPERT_OPTO, 'accuracy')
    if not effs:
        continue

    mean_eff = np.mean(effs)
    sem = np.std(effs, ddof=1) / np.sqrt(len(effs)) if len(effs) > 1 else 0

    # Bootstrap CI from pooled comparison (if available)
    boot = comp.get('boot_ci')
    boot_lo, boot_hi = (np.nan, np.nan)
    if boot is not None and 'accuracy' in boot:
        # This would need n_bootstrap > 0 in phase_pooled_comparison
        pass

    forest_data.append({
        'animal_id': aid, 'mean': mean_eff, 'sem': sem,
        'n_sessions': len(effs), 'genotype': r['genotype'],
    })

# Also add WT
for aid in sorted(wt_reports):
    r = wt_reports[aid]
    effs = _collect_phase_effects(r, OptoPhase.EXPERT_OPTO, 'accuracy')
    if not effs:
        continue
    forest_data.append({
        'animal_id': aid, 'mean': np.mean(effs),
        'sem': np.std(effs, ddof=1) / np.sqrt(len(effs)) if len(effs) > 1 else 0,
        'n_sessions': len(effs), 'genotype': r['genotype'],
    })

if forest_data:
    EQUIV_BOUND = 0.05
    fig, ax = plt.subplots(1, 1, figsize=(7, max(3, 0.5 * len(forest_data))))

    # Equivalence region
    ax.axvspan(-EQUIV_BOUND, EQUIV_BOUND, alpha=0.08, color='green',
               label=f'Equivalence (±{EQUIV_BOUND})')
    ax.axvline(0, ls='-', color='grey', alpha=0.3)

    for i, d in enumerate(forest_data):
        colour = OPTO_COLOUR if d['genotype'] == 'het' else '#888888'
        marker = 'o' if d['genotype'] == 'het' else 's'
        ci95 = 1.96 * d['sem']

        ax.errorbar(d['mean'], i, xerr=ci95,
                    fmt=marker, color=colour, markersize=7,
                    capsize=4, capthick=1, elinewidth=1, alpha=0.8)
        ax.text(d['mean'] + ci95 + 0.005, i,
                f"  {d['animal_id']} (n={d['n_sessions']})",
                va='center', fontsize=8, color='#555555')

    # Cohort mean (het only)
    het_means = [d['mean'] for d in forest_data if d['genotype'] == 'het']
    if het_means:
        grand = np.mean(het_means)
        grand_sem = np.std(het_means, ddof=1) / np.sqrt(len(het_means))
        ax.errorbar(grand, len(forest_data) + 0.5, xerr=1.96 * grand_sem,
                    fmt='D', color='black', markersize=9,
                    capsize=5, capthick=2, elinewidth=2, zorder=5,
                    label=f'Het mean ± 95% CI')

    ax.set_yticks([])
    ax.set_xlabel('Δ accuracy (opto − control)')
    ax.set_title('Per-animal opto effect on accuracy')

    from matplotlib.lines import Line2D
    handles = [
        Line2D([0],[0], marker='o', color='w', markerfacecolor=OPTO_COLOUR,
               markersize=8, label='Het'),
        Line2D([0],[0], marker='s', color='w', markerfacecolor='#888888',
               markersize=8, label='WT'),
    ]
    ax.legend(handles=handles, loc='lower right', fontsize=8)
    fig.tight_layout(); plt.show()
else:
    print("No data for forest plot.")

---
## 4. The punchline — equivalence testing

Animal-level pooling: one mean effect per animal → t-test / TOST.

- **t-test** p < 0.05 → evidence opto does something (bad for null prediction)
- **TOST** p < 0.05 → positive evidence effect is negligible (good for null)
- Neither significant → inconclusive (underpowered)

Bounds: accuracy ±0.05 (5pp), PSE ±0.10, slope/lapse ±0.05, UM RMSE ±0.02.

In [ ]:
EQUIV_BOUNDS = {
    'accuracy': 0.05, 'pse': 0.10,
    'slope': 0.05, 'lapse_low': 0.05, 'lapse_high': 0.05,
}

het_null = {}
for m in params:
    het_null[m] = expert_null_test(het_reports, metric=m,
                                   equivalence_bound=EQUIV_BOUNDS[m])

print(f"{'Metric':>12s} {'n':>4s} {'Mean':>10s} {'SEM':>8s} "
      f"{'t-test p':>10s} {'TOST p':>10s} {'Verdict':>14s}")
print('=' * 72)
for m in params:
    r = het_null[m]
    v = ''
    if 'warning' not in r:
        if r['tost_reject']:
            v = '✓ EQUIVALENT'
        elif r['ttest_p'] < 0.05:
            v = '✗ EFFECT'
        else:
            v = '? inconclusive'
    else:
        v = r['warning'][:20]
    print(f"{m:>12s} {r['n_animals']:>4d} {r['grand_mean']:>+10.4f} "
          f"{r['grand_sem']:>8.4f} {r.get('ttest_p', np.nan):>10.3f} "
          f"{r.get('tost_p', np.nan):>10.3f} {v:>14s}")

In [ ]:
# ── Equivalence plots — all 5 params ────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, m in zip(axes, params):
    plot_equivalence_test(het_null[m], ax=ax,
        title=f'{m} (±{EQUIV_BOUNDS[m]})')
fig.suptitle('Het cohort — TOST equivalence', y=1.04)
fig.tight_layout(); plt.show()

In [ ]:
# ── UM equivalence ──────────────────────────────────────────────────
um_test = expert_um_test(experiment, het_reports, n_bins=8,
                         equivalence_bound=0.02)
v = '✓ EQUIVALENT' if um_test.get('tost_reject', False) else '? inconclusive'
print(f"UM equivalence (Het, n={um_test['n_animals']}): "
      f"RMSE mean={um_test['rmse_mean']:.4f}, "
      f"TOST p={um_test.get('tost_p', np.nan):.3f} → {v}")

In [ ]:
# ── Per-animal UM: control / opto / masking / difference ─────────────
# Same 3-condition pattern as section 1: if an animal has masking
# sessions, show masking UM alongside opto and control.

for aid, data in um_test.get('per_animal', {}).items():
    animal = opto_animals.get(aid)
    if animal is None:
        continue
    ph = all_phases[aid]

    # Get masking sessions for this animal
    a_mask_sess = [s for s, p in zip(animal.sessions, ph) if p == OptoPhase.MASKING]
    has_masking = len(a_mask_sess) > 0

    if has_masking:
        mask_um, _, mask_info = compute_opto_um(a_mask_sess, delta=0)
        n_mask = mask_info.get('n_trials', 0) if isinstance(mask_info, dict) else 0

        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        vmax = max(np.nanmax(np.abs(data['control_um'])),
                   np.nanmax(np.abs(data['opto_um'])),
                   np.nanmax(np.abs(mask_um)), 0.01)

        for ax, um, label in zip(axes[:3], 
            [data['control_um'], data['opto_um'], mask_um],
            ['Control', 'Opto ON', f'Masking (n={n_mask})']):
            im = ax.imshow(um, cmap=UM_CMAP, vmin=-vmax, vmax=vmax,
                           origin='lower', aspect='equal')
            ax.set_title(label); ax.set_xlabel('Previous stimulus')
            plt.colorbar(im, ax=ax, fraction=0.046)
        axes[0].set_ylabel('Current stimulus')

        diff = data['opto_um'] - data['control_um']
        vmax_d = max(np.nanmax(np.abs(diff)), 0.01)
        im = axes[3].imshow(diff, cmap=UM_CMAP, vmin=-vmax_d, vmax=vmax_d,
                            origin='lower', aspect='equal')
        axes[3].set_title('Opto − Control'); axes[3].set_xlabel('Previous stimulus')
        plt.colorbar(im, ax=axes[3], fraction=0.046)
    else:
        fig = plot_opto_um_comparison(
            data['opto_um'], data['control_um'], diff=True)

    fig.suptitle(f"{aid} — RMSE={data['um_rmse']:.3f}, r={data['um_corr']:.3f}",
                 fontsize=12, y=1.02)
    fig.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
# ── WT equivalence (validation) ─────────────────────────────────────
if wt_reports:
    print(f"{'Metric':>12s} {'n':>4s} {'Mean':>10s} {'ttest_p':>10s} "
          f"{'tost_p':>10s} {'Verdict':>14s}")
    print('-' * 65)
    for m in params:
        r = expert_null_test(wt_reports, metric=m,
                             equivalence_bound=EQUIV_BOUNDS[m])
        v = '✓ equiv' if r.get('tost_reject') else '? inconclusive'
        print(f"{m:>12s} {r['n_animals']:>4d} {r['grand_mean']:>+10.4f} "
              f"{r.get('ttest_p', np.nan):>10.3f} "
              f"{r.get('tost_p', np.nan):>10.3f} {v:>14s}")
else:
    print("No WT data.")

In [ ]:
# ── Synthesis ────────────────────────────────────────────────────────
n_equiv = sum(1 for m in params if het_null[m].get('tost_reject', False))
n_effect = sum(1 for m in params
               if het_null[m].get('ttest_p', 1) < 0.05)
um_equiv = um_test.get('tost_reject', False)

print("\n" + "=" * 60)
print("SYNTHESIS")
print("=" * 60)
print(f"  Psychometric params: {n_equiv}/5 show equivalence, "
      f"{n_effect}/5 show significant effect")
print(f"  UM RMSE: {'equivalent' if um_equiv else 'inconclusive'}")
if n_effect == 0 and n_equiv >= 3:
    print("  → Supports dispensability claim.")
elif n_effect > 0:
    print("  → PPC is doing something in expert phase. Revise model.")
else:
    print("  → Underpowered. Need more sessions/animals.")

---
## 5. Between-animal

### 5a. Genotype interaction

In [ ]:
het_eff = [e['effect'] for r in het_reports.values()
           for e in r['within_session']
           if e['phase'] == OptoPhase.EXPERT_OPTO and e['effect'] is not None]
wt_eff  = [e['effect'] for r in wt_reports.values()
           for e in r['within_session']
           if e['phase'] == OptoPhase.EXPERT_OPTO and e['effect'] is not None]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, m in zip(axes, params):
    ix = genotype_interaction(het_eff, wt_eff, metric=m)
    plot_genotype_interaction(ix, ax=ax, title=m)
fig.suptitle('Genotype interaction — Het vs WT (all params)', y=1.03)
fig.tight_layout(); plt.show()

### 5b. Model assignment × opto effects

In [ ]:
try:
    cs = consensus_summary(load_all_assignments(CV_DIR, SBI_STATIC_DIR))
    ar = opto_by_model_assignment(experiment, cs,
        target_phase=OptoPhase.EXPERT_OPTO, metric='accuracy')
    fig = plot_model_assignment_effects(ar['groups'], metric='accuracy',
        comparison=ar['comparison'],
        title='Expert opto effect by model assignment')
    plt.show()
except Exception as e:
    print(f"Consensus not available: {e}")

### 5c. Phase × opto interaction (placeholder — needs post-shift data)

In [ ]:
ix_ph = phase_opto_interaction(reports,
    expert_phase=OptoPhase.EXPERT_OPTO,
    shift_phase=OptoPhase.SHIFT_1_OPTO, metric='accuracy')
if 'warning' in ix_ph:
    print(f"⚠️  {ix_ph['warning']}")
fig = plot_phase_interaction(ix_ph,
    title='Phase × opto interaction (expert vs post-shift)')
plt.show()

---
## 6. Robustness checks

Secondary metrics that may reveal subtle effects even when psychometric
parameters look equivalent.

In [ ]:
# ── Helper: collect paired (opto, control) values per animal ─────────
def _collect_secondary(metric_fn):
    """Collect (animal_id, opto_value, ctrl_value) for each animal."""
    rows = []
    for aid in sorted(opto_animals):
        ph = all_phases[aid]
        o_vals, c_vals = [], []
        for s, p in zip(opto_animals[aid].sessions, ph):
            if p != OptoPhase.EXPERT_OPTO:
                continue
            om, cm = split_trials_by_opto(s)
            ov, cv = metric_fn(s, om, cm)
            if ov is not None:
                o_vals.append(ov)
            if cv is not None:
                c_vals.append(cv)
        if o_vals and c_vals:
            rows.append((aid, np.mean(o_vals), np.mean(c_vals)))
    return rows

In [ ]:
# ── 6a. Stimulus-conditional accuracy ────────────────────────────────
def _stim_cond_acc(session, o_mask, c_mask, threshold=0.5):
    """Return (easy_opto, easy_ctrl, hard_opto, hard_ctrl)."""
    results = {}
    for label, mask in [('opto', o_mask), ('ctrl', c_mask)]:
        arr = extract_trial_arrays(session, mask)
        if arr is None:
            return None
        v = ~arr['no_response']
        stim, ch, cat = arr['stimuli'][v], arr['choices'][v], arr['categories'][v]
        easy = np.abs(stim) >= threshold
        hard = ~easy
        results[f'easy_{label}'] = np.mean(ch[easy] == cat[easy]) if easy.sum() > 3 else np.nan
        results[f'hard_{label}'] = np.mean(ch[hard] == cat[hard]) if hard.sum() > 3 else np.nan
    return results

stim_cond = []
for aid in sorted(opto_animals):
    ph = all_phases[aid]
    easy_o, easy_c, hard_o, hard_c = [], [], [], []
    for s, p in zip(opto_animals[aid].sessions, ph):
        if p != OptoPhase.EXPERT_OPTO:
            continue
        om, cm = split_trials_by_opto(s)
        r = _stim_cond_acc(s, om, cm)
        if r is None:
            continue
        easy_o.append(r['easy_opto']); easy_c.append(r['easy_ctrl'])
        hard_o.append(r['hard_opto']); hard_c.append(r['hard_ctrl'])
    if easy_o:
        stim_cond.append((aid, np.nanmean(easy_o), np.nanmean(easy_c),
                          np.nanmean(hard_o), np.nanmean(hard_c)))

if stim_cond:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, (label, oi, ci) in zip(axes, [
        ('Easy (|s|≥0.5)', 1, 2), ('Hard (|s|<0.5)', 3, 4)]):
        o_vals = [r[oi] for r in stim_cond]
        c_vals = [r[ci] for r in stim_cond]
        for j, (ov, cv) in enumerate(zip(o_vals, c_vals)):
            ax.plot([0, 1], [cv, ov], '-', color='#AAAAAA', lw=0.8)
        ax.scatter(np.zeros(len(c_vals)), c_vals, c=CTRL_COLOUR, s=40,
                   edgecolor='white', lw=0.5, zorder=3, label='Control')
        ax.scatter(np.ones(len(o_vals)), o_vals, c=OPTO_COLOUR, s=40,
                   edgecolor='white', lw=0.5, zorder=3, label='Opto')
        ax.set_xticks([0, 1]); ax.set_xticklabels(['Control', 'Opto'])
        ax.set_ylabel('Accuracy'); ax.set_title(label)
        ax.legend(fontsize=8)
    fig.suptitle('Stimulus-conditional accuracy', y=1.02)
    fig.tight_layout(); plt.show()

In [ ]:
# ── 6b. Reaction time ────────────────────────────────────────────────
def _get_rt(session, o_mask, c_mask):
    if not hasattr(session.trials, 'reaction_time'):
        return None, None
    rt = session.trials.reaction_time
    vo = o_mask & ~np.isnan(rt); vc = c_mask & ~np.isnan(rt)
    return (np.median(rt[vo]) if vo.sum() > 5 else None,
            np.median(rt[vc]) if vc.sum() > 5 else None)

rt_data = _collect_secondary(_get_rt)

# ── 6c. No-response rate ────────────────────────────────────────────
def _get_nr(session, o_mask, c_mask):
    oa = extract_trial_arrays(session, o_mask)
    ca = extract_trial_arrays(session, c_mask)
    ov = oa['no_response'].mean() if oa else None
    cv = ca['no_response'].mean() if ca else None
    return ov, cv

nr_data = _collect_secondary(_get_nr)

# ── 6d. Response bias P(choose B) ───────────────────────────────────
def _get_bias(session, o_mask, c_mask):
    oa = extract_trial_arrays(session, o_mask)
    ca = extract_trial_arrays(session, c_mask)
    ov = np.nanmean(oa['choices']) if oa else None
    cv = np.nanmean(ca['choices']) if ca else None
    return ov, cv

bias_data = _collect_secondary(_get_bias)

# ── Plot all three as paired dot plots ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (data, label, ylabel) in zip(axes, [
    (rt_data, 'Median RT', 'Reaction time (s)'),
    (nr_data, 'No-response rate', 'Proportion'),
    (bias_data, 'Response bias', 'P(choose B)'),
]):
    if not data:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                ha='center', va='center', fontsize=11, color='grey')
        ax.set_title(label)
        continue

    c_vals = [r[2] for r in data]
    o_vals = [r[1] for r in data]
    for cv, ov in zip(c_vals, o_vals):
        ax.plot([0, 1], [cv, ov], '-', color='#AAAAAA', lw=0.8)
    ax.scatter(np.zeros(len(c_vals)), c_vals, c=CTRL_COLOUR, s=40,
               edgecolor='white', lw=0.5, zorder=3)
    ax.scatter(np.ones(len(o_vals)), o_vals, c=OPTO_COLOUR, s=40,
               edgecolor='white', lw=0.5, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Control', 'Opto'])
    ax.set_ylabel(ylabel); ax.set_title(label)

fig.suptitle('Secondary metrics — paired per animal', y=1.02)
fig.tight_layout(); plt.show()

---
## 7. Appendix — full animal reports

One-page summary per animal (phase trajectory + expert stability +
per-session effects) followed by 3-panel UM (control / opto / difference).

In [ ]:
for aid in sorted(opto_animals):
    r = reports[aid]
    animal = opto_animals[aid]
    ph = all_phases[aid]
    print(f"\n{'=' * 60}")
    print(f"{aid} ({r['genotype']})")
    print(f"{'=' * 60}")

    # Standard report figure
    fig = plot_animal_opto_report(animal, r)
    plt.show(); plt.close(fig)

    # 3-condition psychometric: masking / opto ON / control
    a_opto = [s for s, p in zip(animal.sessions, ph) if p == OptoPhase.EXPERT_OPTO]
    a_mask = [s for s, p in zip(animal.sessions, ph) if p == OptoPhase.MASKING]

    if a_opto:
        conds = [
            ('Control', a_opto, 'control', CTRL_COLOUR, '-'),
            ('Opto ON', a_opto, 0, OPTO_COLOUR, '-'),
        ]
        if a_mask:
            conds.append(('Masking', a_mask, 0, '#999999', '--'))

        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        x_fine = np.linspace(-1, 1, 200)
        for label, sl, delta, col, ls in conds:
            stim_all, ch_all = [], []
            for s in sl:
                m = opto_relative_mask(s, delta=delta)
                arr = extract_trial_arrays(s, m)
                if arr is None: continue
                v = ~arr['no_response']
                stim_all.append(arr['stimuli'][v])
                ch_all.append(arr['choices'][v])
            if not stim_all: continue
            stim = np.concatenate(stim_all)
            choice = np.concatenate(ch_all)
            try:
                pf = fit_psychometric(stim, choice)
                y = cumulative_gaussian(x_fine, pf['mu'], pf['sigma'],
                                        pf['lapse_low'], pf['lapse_high'])
                ax.plot(x_fine, y, color=col, ls=ls, lw=1.5,
                        label=f"{label} (n={len(stim)})")
            except Exception: pass
        ax.axhline(0.5, ls=':', color='grey', alpha=0.3)
        ax.axvline(0.0, ls=':', color='grey', alpha=0.3)
        ax.legend(fontsize=7); ax.set_title(f'{aid} — 3-condition psychometric')
        ax.set_xlabel('Stimulus'); ax.set_ylabel('P(choose B)')
        ax.set_xlim(-1.05, 1.05); ax.set_ylim(-0.05, 1.05)
        fig.tight_layout(); plt.show(); plt.close(fig)

    # 3-condition UMs: control / opto / masking / difference
    if a_opto:
        ctrl_um, _, _ = compute_opto_um(a_opto, delta='control')
        opto_um, _, _ = compute_opto_um(a_opto, delta=0)

        if a_mask:
            mask_um, _, mask_info = compute_opto_um(a_mask, delta=0)
            n_mask = mask_info.get('n_trials', 0) if isinstance(mask_info, dict) else 0

            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            vmax = max(np.nanmax(np.abs(ctrl_um)),
                       np.nanmax(np.abs(opto_um)),
                       np.nanmax(np.abs(mask_um)), 0.01)
            for ax, um, lab in zip(axes[:3],
                [ctrl_um, opto_um, mask_um],
                ['Control', 'Opto ON', f'Masking (n={n_mask})']):
                im = ax.imshow(um, cmap=UM_CMAP, vmin=-vmax, vmax=vmax,
                               origin='lower', aspect='equal')
                ax.set_title(lab); ax.set_xlabel('Previous stimulus')
                plt.colorbar(im, ax=ax, fraction=0.046)
            axes[0].set_ylabel('Current stimulus')

            diff = opto_um - ctrl_um
            vmax_d = max(np.nanmax(np.abs(diff)), 0.01)
            im = axes[3].imshow(diff, cmap=UM_CMAP, vmin=-vmax_d, vmax=vmax_d,
                                origin='lower', aspect='equal')
            axes[3].set_title('Opto − Control'); axes[3].set_xlabel('Previous stimulus')
            plt.colorbar(im, ax=axes[3], fraction=0.046)
        else:
            fig = plot_opto_um_comparison(opto_um, ctrl_um, diff=True)

        fig.suptitle(f'{aid} — update matrices', fontsize=12, y=1.02)
        fig.tight_layout(); plt.show(); plt.close(fig)

---
## Summary

The synthesis cell in section 4 gives the headline result. Key things
to check before finalising:

1. **All four controls pass** (masking null, masking≈baseline, WT null, no drift)
2. **TOST rejects** for at least accuracy, PSE, and slope → positive
   evidence for dispensability
3. **No t-test rejects** for any param → no evidence of an effect
4. **UM RMSE equivalent** → opto doesn't change the decision strategy structure

If controls fail or TOST is inconclusive across multiple params,
more sessions or animals are needed before the dispensability
claim can be made.

**Next**: post-shift opto (NB 52, ~2 weeks).